# IOL (Interior O-Line: G + C) round regression (Ridge)

Predict draft round 1–8 (8 = undrafted) using combine + PFF (Pass_Blocking, Run_Blocking) + RAS, arm length, KNN imputation, Ridge regression.
- Train: 2015–2023 (iol_training.csv; RAS and PFF already merged in data_cleaning).
- Test: iol_testing.csv filtered to 2024/2025 (drafted only); 2026 from iol_drafted_2026.csv.

In [213]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# IOL feature set: athletic + PFF derived rates (data_cleaning.py) + penalty + p4
# PFF derived: true_pass_set_pressure_rate = pressures_allowed/snap_counts_pass_block; true_pass_set_sack_rate = sacks_allowed/snap_counts_pass_block
IOL_FEATURES_WITH_COLLEGE = [
    'Height', 'Weight', 'arm_length_inches', 'speed_score', 'agility_score', 'RAS',
    '3Cone', 'Shuttle', 'is_center',
    'true_pass_set_pressure_rate', 'true_pass_set_sack_rate', 'snap_counts_pass_block',
    'snap_counts_run_block', 'grades_run_block', 'gap_rate', 'zone_rate', 'penalty_rate', 'p4_conference'
]
CONTAINS_WITH_COLLEGE_IOL = [
    'contains_height', 'contains_weight', 'contains_arm_length_inches', 'contains_speed_score', 'contains_agility_score', 'contains_ras',
    'contains_3cone', 'contains_shuttle',
    'contains_true_pass_set_pressure_rate', 'contains_true_pass_set_sack_rate', 'contains_snap_counts_pass_block',
    'contains_snap_counts_run_block', 'contains_grades_run_block', 'contains_gap_rate', 'contains_zone_rate', 'contains_penalty_rate', 'contains_p4_conference'
]
FEATURES_WITH_COLLEGE_ALL = IOL_FEATURES_WITH_COLLEGE + CONTAINS_WITH_COLLEGE_IOL

In [214]:
# Load IOL training (2015–2023)
df = pd.read_csv('../data/processed/iol_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print('Train (2015–2023 IOL):', len(df))

Train (2015–2023 IOL): 161


In [215]:
# Print RAS, PFF, and arm length availability
total_count = len(df)
ras_count = df['RAS'].notna().sum()
print(f"Players with RAS: {ras_count} out of {total_count} ({ras_count/total_count*100:.1f}%)")
print(f"Players with true_pass_set_pressure_rate: {df['true_pass_set_pressure_rate'].notna().sum()} out of {total_count}")
print(f"Players with snap_counts_run_block: {df['snap_counts_run_block'].notna().sum()} out of {total_count}")
print(f"Players with arm_length_inches: {df['arm_length_inches'].notna().sum()} out of {total_count}")

Players with RAS: 77 out of 161 (47.8%)
Players with true_pass_set_pressure_rate: 144 out of 161
Players with snap_counts_run_block: 144 out of 161
Players with arm_length_inches: 104 out of 161


In [216]:
def height_inches(h):
    if pd.isna(h): return np.nan
    if isinstance(h, (int, float)) and not (isinstance(h, float) and np.isnan(h)):
        return float(h)
    s = str(h).strip()
    if '-' in s:
        parts = s.split('-')
        return int(parts[0]) * 12 + int(parts[1])
    return np.nan
df['Height'] = df['Height'].apply(height_inches)

df['speed_score'] = np.where(
    df['40yd'].notna() & (df['40yd'] > 0),
    df['Weight'] * 200 / (df['40yd'] ** 4),
    np.nan
)

# Agility score: lower 3Cone/Shuttle = better; z-scores from training, then -(z_3cone + z_shuttle)
mean_3c = df['3Cone'].mean()
std_3c = df['3Cone'].std()
mean_sh = df['Shuttle'].mean()
std_sh = df['Shuttle'].std()
if std_3c == 0 or np.isnan(std_3c): std_3c = 1.0
if std_sh == 0 or np.isnan(std_sh): std_sh = 1.0
z_3 = (df['3Cone'] - mean_3c) / std_3c
z_sh = (df['Shuttle'] - mean_sh) / std_sh
df['agility_score'] = (-z_3.fillna(0)) + (-z_sh.fillna(0))

school_alias = {'Ole Miss': 'Mississippi', 'Miami (FL)': 'Miami', 'Southern California': 'USC', 'Ohio St.': 'Ohio State',
    'Florida St.': 'Florida State', 'Penn St.': 'Penn State', 'NC State': 'North Carolina State', 'Oregon St.': 'Oregon State'}
SEC_SCHOOLS = {'Alabama', 'Arkansas', 'Auburn', 'Florida', 'Georgia', 'Kentucky', 'LSU', 'Mississippi', 'Mississippi State', 'Missouri', 'South Carolina', 'Tennessee', 'Texas A&M', 'Vanderbilt', 'Oklahoma', 'Texas'}
BIG_TEN_SCHOOLS = {'Illinois', 'Indiana', 'Iowa', 'Maryland', 'Michigan', 'Michigan State', 'Minnesota', 'Nebraska', 'Northwestern', 'Ohio State', 'Penn State', 'Purdue', 'Rutgers', 'Wisconsin', 'UCLA', 'USC', 'Oregon', 'Washington'}
BIG_12_SCHOOLS = {'Baylor', 'Iowa State', 'Kansas', 'Kansas State', 'Oklahoma State', 'TCU', 'Texas Tech', 'West Virginia', 'BYU', 'UCF', 'Cincinnati', 'Houston', 'Arizona', 'Arizona State', 'Colorado', 'Utah'}
ACC_SCHOOLS = {'Boston College', 'Clemson', 'Duke', 'Florida State', 'Georgia Tech', 'Louisville', 'Miami', 'North Carolina', 'North Carolina State', 'NC State', 'Pittsburgh', 'Syracuse', 'Virginia', 'Virginia Tech', 'Wake Forest', 'California', 'SMU', 'Stanford'}
PAC12_SCHOOLS = {'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon', 'Oregon State', 'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State'}
P4_SCHOOLS = SEC_SCHOOLS | BIG_TEN_SCHOOLS | BIG_12_SCHOOLS | ACC_SCHOOLS | PAC12_SCHOOLS
P4_SCHOOLS_NO_PAC12 = SEC_SCHOOLS | BIG_TEN_SCHOOLS | BIG_12_SCHOOLS | ACC_SCHOOLS

def is_p4(row):
    s = row.get('School')
    if pd.isna(s) or s == '': return 0
    sn = school_alias.get(s, s)
    year = row.get('Year', 2023)
    schools = P4_SCHOOLS if year <= 2023 else P4_SCHOOLS_NO_PAC12
    return 1 if sn in schools else 0
df['p4_conference'] = df.apply(is_p4, axis=1)

df['is_center'] = (df['Pos'].astype(str).str.strip().str.upper() == 'C').astype(int) if 'is_center' not in df.columns else df['is_center']

df['contains_height'] = df['Height'].notna().astype(int)
df['contains_weight'] = df['Weight'].notna().astype(int)
df['contains_arm_length_inches'] = df['arm_length_inches'].notna().astype(int) if 'arm_length_inches' in df.columns else 0
df['arm_33_plus'] = (df['arm_length_inches'] >= 33).fillna(False).astype(int) if 'arm_length_inches' in df.columns else 0
df['arm_34_plus'] = (df['arm_length_inches'] >= 34).fillna(False).astype(int) if 'arm_length_inches' in df.columns else 0
df['contains_speed_score'] = df['speed_score'].notna().astype(int)
df['contains_agility_score'] = (df['3Cone'].notna() | df['Shuttle'].notna()).astype(int)
df['contains_3cone'] = df['3Cone'].notna().astype(int)
df['contains_shuttle'] = df['Shuttle'].notna().astype(int)
df['contains_ras'] = df['RAS'].notna().astype(int)
df['contains_true_pass_set_pressure_rate'] = df['true_pass_set_pressure_rate'].notna().astype(int) if 'true_pass_set_pressure_rate' in df.columns else 0
df['contains_true_pass_set_sack_rate'] = df['true_pass_set_sack_rate'].notna().astype(int) if 'true_pass_set_sack_rate' in df.columns else 0
df['contains_snap_counts_pass_block'] = df['snap_counts_pass_block'].notna().astype(int) if 'snap_counts_pass_block' in df.columns else 0
df['contains_snap_counts_run_block'] = df['snap_counts_run_block'].notna().astype(int) if 'snap_counts_run_block' in df.columns else 0
df['contains_grades_run_block'] = df['grades_run_block'].notna().astype(int) if 'grades_run_block' in df.columns else 0
df['contains_gap_rate'] = df['gap_rate'].notna().astype(int) if 'gap_rate' in df.columns else 0
df['contains_zone_rate'] = df['zone_rate'].notna().astype(int) if 'zone_rate' in df.columns else 0
df['contains_penalty_rate'] = df['penalty_rate'].notna().astype(int) if 'penalty_rate' in df.columns else 0
df['contains_p4_conference'] = df['School'].notna().astype(int)

In [217]:
y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_WITH_COLLEGE_ALL].copy()
imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)
y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
print('Train MAE (round 1–8):', round(mean_absolute_error(y, y_pred_train), 4))
print('Train samples:', len(y))

Train MAE (round 1–8): 1.7585
Train samples: 161


In [218]:
def prepare_iol_df(ldf, year):
    ldf = ldf.copy()
    ldf['Year'] = year
    for col in ['Height', 'Weight', '40yd', 'Vertical', 'Bench', 'Broad Jump', '3Cone', 'Shuttle', 'RAS', 'Pos'] + IOL_FEATURES_WITH_COLLEGE:
        if col not in ldf.columns:
            ldf[col] = np.nan
    if ldf['Height'].dtype == object or (ldf['Height'].astype(str).str.contains('-', na=False).any()):
        ldf['Height'] = ldf['Height'].apply(height_inches)
    else:
        ldf['Height'] = pd.to_numeric(ldf['Height'], errors='coerce')
    ldf['speed_score'] = np.where(ldf['40yd'].notna() & (ldf['40yd'] > 0), ldf['Weight'] * 200 / (ldf['40yd'] ** 4), np.nan)
    z_3 = (ldf['3Cone'] - mean_3c) / std_3c
    z_sh = (ldf['Shuttle'] - mean_sh) / std_sh
    ldf['agility_score'] = (-z_3.fillna(0)) + (-z_sh.fillna(0))
    ldf['p4_conference'] = ldf.apply(is_p4, axis=1)
    ldf['is_center'] = (ldf['Pos'].astype(str).str.strip().str.upper() == 'C').astype(int) if 'Pos' in ldf.columns else 0
    ldf['contains_height'] = ldf['Height'].notna().astype(int)
    ldf['contains_weight'] = ldf['Weight'].notna().astype(int)
    ldf['contains_arm_length_inches'] = ldf['arm_length_inches'].notna().astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['arm_33_plus'] = (ldf['arm_length_inches'] >= 33).fillna(False).astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['arm_34_plus'] = (ldf['arm_length_inches'] >= 34).fillna(False).astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['contains_speed_score'] = ldf['speed_score'].notna().astype(int)
    ldf['contains_agility_score'] = (ldf['3Cone'].notna() | ldf['Shuttle'].notna()).astype(int)
    ldf['contains_3cone'] = ldf['3Cone'].notna().astype(int)
    ldf['contains_shuttle'] = ldf['Shuttle'].notna().astype(int)
    ldf['contains_ras'] = ldf['RAS'].notna().astype(int)
    ldf['contains_true_pass_set_pressure_rate'] = ldf['true_pass_set_pressure_rate'].notna().astype(int) if 'true_pass_set_pressure_rate' in ldf.columns else 0
    ldf['contains_true_pass_set_sack_rate'] = ldf['true_pass_set_sack_rate'].notna().astype(int) if 'true_pass_set_sack_rate' in ldf.columns else 0
    ldf['contains_snap_counts_pass_block'] = ldf['snap_counts_pass_block'].notna().astype(int) if 'snap_counts_pass_block' in ldf.columns else 0
    ldf['contains_snap_counts_run_block'] = ldf['snap_counts_run_block'].notna().astype(int) if 'snap_counts_run_block' in ldf.columns else 0
    ldf['contains_grades_run_block'] = ldf['grades_run_block'].notna().astype(int) if 'grades_run_block' in ldf.columns else 0
    ldf['contains_gap_rate'] = ldf['gap_rate'].notna().astype(int) if 'gap_rate' in ldf.columns else 0
    ldf['contains_zone_rate'] = ldf['zone_rate'].notna().astype(int) if 'zone_rate' in ldf.columns else 0
    ldf['contains_penalty_rate'] = ldf['penalty_rate'].notna().astype(int) if 'penalty_rate' in ldf.columns else 0
    ldf['contains_p4_conference'] = ldf['School'].notna().astype(int)
    return ldf

iol_testing = pd.read_csv('../data/processed/iol_testing.csv')
iol_2024 = prepare_iol_df(iol_testing[iol_testing['Year'] == 2024], 2024)
iol_2025 = prepare_iol_df(iol_testing[iol_testing['Year'] == 2025], 2025)
X_24_raw = iol_2024[FEATURES_WITH_COLLEGE_ALL].copy()
X_25_raw = iol_2025[FEATURES_WITH_COLLEGE_ALL].copy()
X_24 = imputer.transform(X_24_raw)
X_25 = imputer.transform(X_25_raw)
X_24_scaled = scaler.transform(X_24)
X_25_scaled = scaler.transform(X_25)
pred_24 = np.clip(ridge.predict(X_24_scaled), 1, 8)
pred_25 = np.clip(ridge.predict(X_25_scaled), 1, 8)
actual_24 = iol_2024['Round'].astype(int).values
actual_25 = iol_2025['Round'].astype(int).values

def eval_metrics(actual, pred, label):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    exact = (np.round(pred) == actual).mean()
    within_1 = (np.abs(np.round(pred) - actual) <= 1).mean()
    print(f'{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, Exact={exact:.2%}, Within-1={within_1:.2%}')

print('2024 IOL:')
eval_metrics(actual_24, pred_24, '2024')
print('2025 IOL:')
eval_metrics(actual_25, pred_25, '2025')

2024 IOL:
2024 (n=22): MAE=1.4025, RMSE=1.7463, R²=0.0659, Exact=27.27%, Within-1=59.09%
2025 IOL:
2025 (n=10): MAE=1.5106, RMSE=1.6849, R²=0.2201, Exact=10.00%, Within-1=50.00%


In [219]:
def pred_round_to_tier(p):
    if p < 1.5: return ('1st', 'Elite')
    if p < 2.5: return ('2nd', 'Day 2')
    if p < 4.5: return ('3rd-4th', 'Day 2')
    if p < 6.5: return ('5th-6th', 'Day 3')
    if p < 7.5: return ('7th', 'Day 3')
    return ('UDFA', 'Undrafted')

iol_2024_display = iol_2024[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
iol_2024_display['predicted_round'] = pred_24
iol_2024_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_24]
iol_2025_display = iol_2025[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
iol_2025_display['predicted_round'] = pred_25
iol_2025_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_25]
print('2024 drafted IOL')
display(iol_2024_display.sort_values('predicted_round'))
print('2025 drafted IOL')
display(iol_2025_display.sort_values('predicted_round'))

2024 drafted IOL


,Round,Pick,Player,School,Year,predicted_round,tier_label
16,4.0,117.0,Tanor Bortolini,Wisconsin,2024,3.644909,3rd-4th
12,2.0,44.0,Jackson Powers-Johnson,Oregon,2024,4.017779,3rd-4th
15,7.0,217.0,Beaux Limmer,Arkansas,2024,4.710442,5th-6th
18,5.0,159.0,Hunter Nourzad,Penn State,2024,4.853739,5th-6th
14,5.0,141.0,Sedrick Van Pran,Georgia,2024,4.855348,5th-6th
3,3.0,81.0,Christian Haynes,UConn,2024,4.951963,5th-6th
6,4.0,103.0,Layden Robinson,Texas A&M,2024,5.265211,5th-6th
13,2.0,51.0,Zach Frazier,W VIRGINIA,2024,5.282034,5th-6th
0,3.0,73.0,Cooper Beebe,Kansas State,2024,5.330585,5th-6th
20,6.0,190.0,Dylan McMahon,NC STATE,2024,5.360188,5th-6th


2025 drafted IOL


,Round,Pick,Player,School,Year,predicted_round,tier_label
23,3.0,95.0,Jared Wilson,Georgia,2025,2.652468,3rd-4th
29,7.0,249.0,Connor Colby,Iowa,2025,3.907646,3rd-4th
22,2.0,57.0,Tate Ratledge,Georgia,2025,4.299912,3rd-4th
24,3.0,89.0,Wyatt Milum,West Virginia,2025,4.352382,3rd-4th
28,3.0,98.0,Caleb Rogers,Texas Tech,2025,4.518457,5th-6th
26,3.0,81.0,Dylan Fairchild,Georgia,2025,4.897741,5th-6th
31,6.0,211.0,Hayden Conner,Texas,2025,4.984397,5th-6th
27,7.0,221.0,Jonah Monheim,USC,2025,6.105755,5th-6th
25,5.0,171.0,Miles Frazier,LSU,2025,6.687418,7th
30,7.0,243.0,Garrett Dellinger,LSU,2025,8.000000,UDFA


In [220]:
# 2026 from iol_drafted_2026.csv
iol_2026 = prepare_iol_df(pd.read_csv('iol_drafted_2026.csv'), 2026)
if len(iol_2026) == 0:
    print('No 2026 IOL in iol_drafted_2026.csv')
else:
    X_26_raw = iol_2026[FEATURES_WITH_COLLEGE_ALL].copy()
    X_26 = imputer.transform(X_26_raw)
    X_26_scaled = scaler.transform(X_26)
    pred_26 = np.clip(ridge.predict(X_26_scaled), 1, 8)
    display_cols = [c for c in ['Round', 'Pick', 'Player', 'School', 'Year'] if c in iol_2026.columns]
    iol_2026_display = iol_2026[display_cols].copy()
    iol_2026_display['predicted_round'] = pred_26
    iol_2026_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_26]
    iol_2026_display[['Player','School','predicted_round']].assign(Pos='IOL').to_csv('../data/processed/iol_2026_predictions.csv', index=False)
    print(f'2026 IOL (n={len(pred_26)}): Predictions generated')
    display(iol_2026_display.sort_values('predicted_round'))

2026 IOL (n=24): Predictions generated


,Player,School,Year,predicted_round,tier_label
16,Logan Jones,Iowa,2026,4.651803,5th-6th
18,Matt Gulbin,Wake Forest,2026,5.352149,5th-6th
23,Zach Rice,North Carolina,2026,5.709568,5th-6th
12,Jake Slaughter,Florida,2026,5.930198,5th-6th
20,Bryce Foster,Texas A&M,2026,5.960647,5th-6th
22,Raheem Anderson II,Michigan,2026,6.082437,5th-6th
19,Pat Coogan,Notre Dame,2026,6.382396,5th-6th
15,Parker Brailsford,Washington,2026,6.621236,7th
21,Jordan White,West Virginia,2026,6.720354,7th
13,Connor Lew,Auburn,2026,6.819023,7th


In [221]:
# Model results on entire training set (2017–2023), ordered by predicted_round
train_display = df[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
train_display['predicted_round'] = y_pred_train
train_display['tier_label'] = [pred_round_to_tier(x)[0] for x in y_pred_train]
train_display['interpretation'] = [pred_round_to_tier(x)[1] for x in y_pred_train]
train_display = train_display.sort_values('predicted_round').reset_index(drop=True)
train_display

,Round,Pick,Player,School,Year,predicted_round,tier_label,interpretation
0,2.0,39.0,James Daniels,Iowa,2018,1.550056,2nd,Day 2
1,1.0,6.0,Quenton Nelson,Notre Dame,2018,2.305900,2nd,Day 2
2,1.0,25.0,Tyler Linderbaum,Iowa,2022,2.602139,3rd-4th,Day 2
3,4.0,140.0,Zach Tom,Wake Forest,2022,2.648572,3rd-4th,Day 2
4,3.0,79.0,Isaac Seumalo,Oregon State,2016,2.887108,3rd-4th,Day 2
...,...,...,...,...,...,...,...,...
156,NaN,NaN,Avery Gennesy,Texas A&M,2017,7.784372,UDFA,Undrafted
157,NaN,NaN,TJ Bass,Oregon,2023,7.785722,UDFA,Undrafted
158,NaN,NaN,Damien Mama,USC,2017,8.000000,UDFA,Undrafted
159,NaN,NaN,K.J. Malone,LSU,2018,8.000000,UDFA,Undrafted
